# Tree BPE demo

This gives basic syntax and timing benchmarks for `TreeBytePairEncodingVectorizer`.

Trees are represented as pairs

```python
(adjacency_matrix, label_sequence)
```

where `adjacency_matrix` is a SciPy sparse parent-to-child adjacency matrix and `label_sequence` is a one-dimensional sequence of string labels. 

In [ ]:
import time
from collections import deque

import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt

from vectorizers import TreeBytePairEncodingVectorizer

plt.rcParams["figure.dpi"] = 120

## Synthetic labelled trees

The generator below creates moderately sized rooted trees with a small label alphabet and repeated local motifs, so as to create visually-meaningful contractions.

In [ ]:
TRANSITIONS = {
    "root":   (["block", "block", "call"], [0.55, 0.35, 0.10]),
    "block":  (["assign", "if", "call", "return"], [0.35, 0.25, 0.25, 0.15]),
    "if":     (["op", "block", "block", "name"], [0.25, 0.35, 0.25, 0.15]),
    "assign": (["name", "op", "call", "num"], [0.45, 0.25, 0.20, 0.10]),
    "call":   (["name", "arg", "arg", "num"], [0.40, 0.30, 0.20, 0.10]),
    "return": (["name", "call", "num"], [0.35, 0.35, 0.30]),
    "op":     (["name", "num", "name"], [0.35, 0.35, 0.30]),
    "arg":    (["name", "num", "call"], [0.45, 0.35, 0.20]),
    "name":   (["arg", "num"], [0.30, 0.70]),
    "num":    (["arg", "name"], [0.50, 0.50]),
}

MEAN_CHILDREN = {
    "root": 3.0,
    "block": 3.2,
    "if": 2.6,
    "assign": 2.0,
    "call": 2.6,
    "return": 1.4,
    "op": 2.0,
    "arg": 1.0,
    "name": 0.15,
    "num": 0.05,
}


def choose_child_label(parent_label, rng):
    choices, probs = TRANSITIONS[parent_label]
    return str(rng.choice(choices, p=probs))


def planned_child_count(label, remaining, rng):
    """Draw a readable number of children for one synthetic node."""
    if remaining <= 0:
        return 0

    count = int(rng.poisson(MEAN_CHILDREN[label]))

    # Keep the visible trees from becoming too skinny.
    if label in {"root", "block"}:
        count = max(1, count)

    # Very high-degree nodes are hard to read in the plot.
    return min(count, remaining, 5)


def simulate_labelled_tree(n_nodes=80, rng=None):
    """Return one labelled directed tree as (sparse_adjacency, labels)."""
    if rng is None:
        rng = np.random.default_rng()

    labels = ["root"]
    rows = []
    cols = []

    # Breadth-ish expansion gives more readable depths than uniformly attaching
    # every new node to a random existing node.
    frontier = [0]
    cursor = 0

    while len(labels) < n_nodes:
        if cursor >= len(frontier):
            # If a branch dies out, attach to an existing non-leaf-ish label.
            candidates = [i for i, label in enumerate(labels) if label not in {"name", "num"}]
            parent = int(rng.choice(candidates if candidates else np.arange(len(labels))))
        else:
            parent = frontier[cursor]
            cursor += 1

        remaining = n_nodes - len(labels)
        n_children = planned_child_count(labels[parent], remaining, rng)
        if n_children == 0 and cursor >= len(frontier):
            n_children = 1

        for _ in range(n_children):
            if len(labels) >= n_nodes:
                break
            child = len(labels)
            labels.append(choose_child_label(labels[parent], rng))
            rows.append(parent)
            cols.append(child)
            frontier.append(child)

    adjacency = sparse.csr_matrix(
        (np.ones(len(rows), dtype=np.float32), (rows, cols)),
        shape=(n_nodes, n_nodes),
        dtype=np.float32,
    )
    return adjacency, np.asarray(labels, dtype=object)


def simulate_tree_collection(n_trees, n_nodes=80, seed=0):
    rng = np.random.default_rng(seed)
    return [simulate_labelled_tree(n_nodes=n_nodes, rng=rng) for _ in range(n_trees)]


def tree_size(tree):
    adjacency, labels = tree
    return len(labels), int(adjacency.nnz)

## Fit tree BPE on a synthetic collection


In [ ]:
training_trees = simulate_tree_collection(n_trees=300, n_nodes=80, seed=20260701)

bpe = TreeBytePairEncodingVectorizer(
    max_vocab_size=14,
    min_pair_count=30,
    return_type="tokens",
)

encoded_training_trees = bpe.fit_transform(training_trees)

print(f"training trees: {len(training_trees)}")
print(f"nodes per training tree: {tree_size(training_trees[0])[0]}")
print(f"learned BPE rules: {len(bpe.rules_)}")

rules_table = pd.DataFrame(
    {
        "rank": rule.rank,
        "token": rule.token.replace("__tree_bpe_", "T").replace("__", ""),
        "parent": str(rule.parent_label),
        "child": str(rule.child_label),
        "raw_count_at_selection": rule.count,
        "actual_contractions": rule.actual_events,
    }
    for rule in bpe.rules_
)

rules_table

## A small tree plotter

Some helper functions for showing the results.

In [ ]:
def children_from_adjacency(adjacency):
    adj = adjacency.tocoo() if sparse.issparse(adjacency) else sparse.coo_matrix(adjacency)
    children = [[] for _ in range(adj.shape[0])]
    for parent, child in sorted(zip(adj.row.tolist(), adj.col.tolist())):
        children[int(parent)].append(int(child))
    return children


def root_from_adjacency(adjacency):
    adj = adjacency.tocsc() if sparse.issparse(adjacency) else sparse.csc_matrix(adjacency)
    indegree = np.asarray(adj.sum(axis=0)).ravel()
    roots = np.flatnonzero(indegree == 0)
    if len(roots) != 1:
        raise ValueError(f"expected exactly one root; found {len(roots)}")
    return int(roots[0])


def tree_levels(adjacency):
    children = children_from_adjacency(adjacency)
    root = root_from_adjacency(adjacency)
    levels = np.zeros(len(children), dtype=int)
    queue = deque([root])
    while queue:
        parent = queue.popleft()
        for child in children[parent]:
            levels[child] = levels[parent] + 1
            queue.append(child)
    return levels


def tidy_tree_layout(adjacency):
    children = children_from_adjacency(adjacency)
    root = root_from_adjacency(adjacency)
    levels = tree_levels(adjacency)

    x = np.zeros(len(children), dtype=float)
    next_leaf_x = 0

    def assign_x(node):
        nonlocal next_leaf_x
        if not children[node]:
            x[node] = next_leaf_x
            next_leaf_x += 1
            return x[node]
        child_positions = [assign_x(child) for child in children[node]]
        x[node] = float(np.mean(child_positions))
        return x[node]

    assign_x(root)
    y = -levels.astype(float)
    return x, y, children, levels


def short_label(label):
    label = str(label)
    if label.startswith("__tree_bpe_") and label.endswith("__"):
        return "T" + label[len("__tree_bpe_") : -len("__")]
    abbreviations = {
        "assign": "asg",
        "return": "ret",
        "block": "blk",
        "name": "nam",
        "num": "num",
        "root": "root",
        "call": "call",
        "arg": "arg",
        "if": "if",
        "op": "op",
    }
    return abbreviations.get(label, label[:6])


def plot_labelled_tree(tree, title=None, ax=None, show_node_ids=False):
    adjacency, labels = tree
    labels = np.asarray(labels, dtype=object)
    x, y, children, levels = tidy_tree_layout(adjacency)

    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    for parent, child_list in enumerate(children):
        for child in child_list:
            ax.plot([x[parent], x[child]], [y[parent], y[child]], linewidth=1, zorder=1)

    ax.scatter(x, y, s=850, zorder=2)

    for node, label in enumerate(labels):
        text = short_label(label)
        if show_node_ids:
            text = f"{node}\n{text}"
        ax.text(x[node], y[node], text, ha="center", va="center", fontsize=8, zorder=3)

    n_nodes = len(labels)
    depth = int(levels.max()) if n_nodes else 0
    ax.set_title(f"{title or ''}\n{n_nodes} nodes, depth {depth}")
    ax.set_axis_off()

    # Add a little breathing room around the drawing.
    if len(x):
        ax.set_xlim(x.min() - 1.0, x.max() + 1.0)
        ax.set_ylim(y.min() - 0.8, y.max() + 0.8)
    return ax

## Encode and display a new tree

The fitted encoder is applied to a new synthetic tree. The condensed tree is the actual transformed tree returned by `return_type="tokens"`; BPE tokens are shown as `T0`, `T1`, and so on.

In [ ]:
new_tree = simulate_labelled_tree(n_nodes=48, rng=np.random.default_rng(369))
condensed_tree = bpe.transform([new_tree])[0]

original_nodes = len(new_tree[1])
condensed_nodes = len(condensed_tree[1])
print(f"original nodes:  {original_nodes}")
print(f"condensed nodes: {condensed_nodes}")
print(f"node reduction:  {original_nodes - condensed_nodes} ({100 * (original_nodes - condensed_nodes) / original_nodes:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)
plot_labelled_tree(new_tree, title="Original tree", ax=axes[0])
plot_labelled_tree(condensed_tree, title="After fitted tree BPE", ax=axes[1])
plt.show()

In [ ]:
# Token legend for the displayed BPE labels.
rules_table[["token", "parent", "child", "actual_contractions"]]

## Timing test: fit and transform as the number of trees grows

This section measures rough wall-clock times for trees of size about 100.

In [ ]:
def benchmark_tree_bpe(
    tree_counts=(100, 300, 1_000, 3_000, 10_000),
    n_nodes=100,
    max_vocab_size=16,
    min_pair_count=25,
    seed=12345,
):
    rows = []

    for n_trees in tree_counts:
        print(f"n_trees={n_trees}: generating data")
        train_trees = simulate_tree_collection(
            n_trees=n_trees,
            n_nodes=n_nodes,
            seed=seed + n_trees,
        )
        test_trees = simulate_tree_collection(
            n_trees=n_trees,
            n_nodes=n_nodes,
            seed=seed + 1_000_000 + n_trees,
        )

        vectorizer = TreeBytePairEncodingVectorizer(
            max_vocab_size=max_vocab_size,
            min_pair_count=min_pair_count,
            return_type="matrix",
        )

        print(f"n_trees={n_trees}: fitting")
        start = time.perf_counter()
        vectorizer.fit(train_trees)
        fit_seconds = time.perf_counter() - start

        print(f"n_trees={n_trees}: transforming")
        start = time.perf_counter()
        transformed = vectorizer.transform(test_trees)
        transform_seconds = time.perf_counter() - start

        rows.append(
            {
                "n_trees": n_trees,
                "n_nodes_per_tree": n_nodes,
                "fit_seconds": fit_seconds,
                "transform_seconds": transform_seconds,
                "fit_ms_per_tree": 1000 * fit_seconds / n_trees,
                "transform_ms_per_tree": 1000 * transform_seconds / n_trees,
                "n_rules": len(vectorizer.rules_),
                "matrix_shape": transformed.shape,
                "matrix_nnz": transformed.nnz,
            }
        )
        print(
            f"n_trees={n_trees}: fit={fit_seconds:.3f}s, "
            f"transform={transform_seconds:.3f}s, rules={len(vectorizer.rules_)}"
        )

    return pd.DataFrame(rows)


TREE_COUNTS = (100, 300, 1_000, 3_000, 10_000)
timing = benchmark_tree_bpe(tree_counts=TREE_COUNTS, n_nodes=100)
timing

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(timing["n_trees"], timing["fit_seconds"], marker="o", label="fit")
ax.plot(timing["n_trees"], timing["transform_seconds"], marker="o", label="transform")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of trees")
ax.set_ylabel("seconds")
ax.set_title("Tree BPE timing on synthetic trees of about 100 nodes")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(timing["n_trees"], timing["fit_ms_per_tree"], marker="o", label="fit")
ax.plot(timing["n_trees"], timing["transform_ms_per_tree"], marker="o", label="transform")
ax.set_xscale("log")
ax.set_xlabel("number of trees")
ax.set_ylabel("milliseconds per tree")
ax.set_title("Per-tree timing")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()